# 11 · 优化器：SGD / Momentum / RMSProp / Adam

> **本节属于 Part 5 · 优化与训练工程。**

前面我们一直手写 `p.data -= lr * p.grad`（朴素 SGD）。但朴素 SGD 有不少毛病：在狭长的"山谷"里来回震荡、对学习率敏感、容易卡在平坦区。本节我们实现并对比四种主流优化器，并把它们封装成 `minitorch.optim`——从此训练循环只需 `opt.step()`。

## 学习目标

- 理解 **Momentum**（动量）、**RMSProp**（自适应步长）、**Adam**（两者结合 + 偏差校正）的思想
- 实现 `Optimizer` 基类与四种优化器
- 在 2D 损失面上**可视化对比**它们的优化轨迹

## 直觉与原理

- **SGD**：$\theta \leftarrow \theta - \eta\, g$。简单，但在各方向曲率差异大时会震荡。
- **Momentum**：累积"速度" $v \leftarrow \beta v + g$，再 $\theta \leftarrow \theta - \eta v$。像小球滚下山，能冲过小坑、抑制震荡。
- **RMSProp**：用梯度平方的滑动平均 $s$ 自适应缩放步长 $\theta \leftarrow \theta - \eta\, g/\sqrt{s}$，让每个参数有自己的"有效学习率"。
- **Adam**：= Momentum + RMSProp + **偏差校正**，深度学习里最常用的默认选择。

看 SGD 与 Adam 的真实实现：

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from minitorch import Tensor
from minitorch.optim import SGD, RMSProp, Adam

print(inspect.getsource(SGD.step))
print(inspect.getsource(Adam.step))

## 在 2D 损失面上对比

考虑一个狭长的碗 $f(x,y) = 0.1x^2 + 2y^2$（$y$ 方向陡、$x$ 方向缓）。这种"病态"曲率最能暴露优化器的差异。我们让每个优化器从同一点出发，记录轨迹。

In [ ]:
def optimize(OptClass, steps=50, start=(-4.5, -4.5), **kw):
    p = Tensor(np.array(start))
    scale = Tensor(np.array([0.1, 2.0]))
    opt = OptClass([p], **kw)
    traj = [p.data.copy()]
    for _ in range(steps):
        opt.zero_grad()
        (p * p * scale).sum().backward()      # f = 0.1 x^2 + 2 y^2
        opt.step()
        traj.append(p.data.copy())
    return np.array(traj)

trajs = {
    "SGD (lr=0.1)":          optimize(SGD, lr=0.1),
    "Momentum (lr=0.05)":    optimize(SGD, lr=0.05, momentum=0.9),
    "RMSProp (lr=0.3)":      optimize(RMSProp, lr=0.3),
    "Adam (lr=0.5)":         optimize(Adam, lr=0.5),
}
for name, t in trajs.items():
    print(f"{name:22s} 终点 = ({t[-1,0]:+.3f}, {t[-1,1]:+.3f})   （最优在 0,0）")

In [ ]:
xs = np.linspace(-5, 5, 200); ys = np.linspace(-5, 5, 200)
XX, YY = np.meshgrid(xs, ys); ZZ = 0.1 * XX**2 + 2 * YY**2
plt.figure(figsize=(7, 5.5))
plt.contour(XX, YY, ZZ, levels=25, cmap="gray", alpha=0.4)
for name, t in trajs.items():
    plt.plot(t[:, 0], t[:, 1], marker=".", ms=4, label=name)
plt.scatter([0], [0], c="red", marker="*", s=150, zorder=5, label="optimum")
plt.legend(fontsize=8); plt.title("Optimizer trajectories on f = 0.1x^2 + 2y^2")
plt.xlabel("x"); plt.ylabel("y"); plt.tight_layout(); plt.show()

你应能看到：朴素 **SGD** 在陡峭的 $y$ 方向来回震荡、在平缓的 $x$ 方向爬得很慢；**Momentum** 抑制了震荡；**RMSProp/Adam** 因为自适应步长，更快、更直地冲向最优点。

## PyTorch 对照

`torch.optim` 里的同名优化器在同一问题上行为一致。

In [ ]:
import torch

def optimize_torch(make_opt, steps=50, start=(-4.5, -4.5)):
    p = torch.tensor(list(start), requires_grad=True)
    scale = torch.tensor([0.1, 2.0])
    opt = make_opt([p])
    traj = [p.detach().numpy().copy()]
    for _ in range(steps):
        opt.zero_grad()
        (p * p * scale).sum().backward()
        opt.step()
        traj.append(p.detach().numpy().copy())
    return np.array(traj)

t_adam = optimize_torch(lambda ps: torch.optim.Adam(ps, lr=0.5))
print("PyTorch Adam 终点:", np.round(t_adam[-1], 3))
print("minitorch Adam 终点:", np.round(trajs["Adam (lr=0.5)"][-1], 3))

## 📦 沉淀进 minitorch

`Optimizer / SGD / RMSProp / Adam` 在 **`minitorch/optim/`**，由 `tests/test_optim.py` 守护。从下一节起，训练循环就用 `opt.zero_grad()` / `opt.step()`。

## 小练习

1. **学习率与动量**：调大 Momentum 的 `lr` 或 `momentum`，观察它是否会"冲过头"来回振荡。
2. **加一个鞍点**：把损失换成 $f=x^2 - y^2$（鞍点在原点），从靠近 $y$ 轴处出发，看哪些优化器更容易逃离鞍点。
3. **Adam 的偏差校正**：把 `Adam.step` 里的偏差校正去掉（不除以 $1-\beta^t$），观察训练初期是否变慢。

## 小结 & 下一站

✅ 我们实现并对比了四种优化器，理解了 Momentum/自适应步长为什么有用，并把它们封装进 `minitorch.optim`。

**下一站 → `12_dataloader_and_dataset`**：把"切 mini-batch、打乱数据"也封装成 `Dataset / DataLoader`，让训练循环彻底清爽——然后用 `DataLoader + Adam` 重新训练 MNIST。